In [59]:
using LinearAlgebra
using Random
using Statistics
using Printf

### Problem 1.


In [6]:
# a) Implement the clgs algorithm for an m-by-n real matrix A, m ≥ n; the pseudo-code is given in Algorithm 7.1. 
# The function should output matrices Q and R.
# Solution: Note that here, the algorithm computes the reduced QR since, we can always get to Full QR by adding more
# orthogonal columns to Q and simply adding a zero block to R.

function clgs(A::AbstractMatrix{<:Real})
    m, n = size(A)
    Q̂ = zeros(Float64, m, n)
    R̂ = zeros(Float64, n, n)
    for j in 1:n
        v = Float64.(A[:, j])
        for i in 1:(j-1)
            R̂[i, j] = dot(Q̂[:, i], A[:, j])
            v -= R̂[i, j] * Q̂[:, i]
        end
        R̂[j, j] = norm(v)
        Q̂[:, j] = v / R̂[j, j]
    end
    return Q̂, R̂
end

clgs (generic function with 1 method)

In [ ]:
#Test Block for algorithm correctness:

Random.seed!(3)
A = Float64.(rand(-5:5, 4, 3))
display(A)
Q, R = clgs(A)
Ā = Q * R
ϵ = Ā - A

display("Q:")
display(Q)
display("R:")
display(R)
display("Ā = QR:")
display(Ā)
display("ϵ = Ā - A")
display(ϵ) #Some Floating Point Error is Noticed where the matrix entry was originally zero, rest matches original.

4×3 Matrix{Float64}:
 -2.0  2.0   3.0
  5.0  0.0   0.0
  5.0  5.0  -2.0
 -4.0  3.0   5.0

"Q:"

4×3 Matrix{Float64}:
 -0.239046   0.371863   0.282791
  0.597614  -0.10591    0.793219
  0.597614   0.717836  -0.356111
 -0.478091   0.578976   0.404991

"R:"

3×3 Matrix{Float64}:
 8.3666  1.07571  -4.30282
 0.0     6.06983   2.57479
 0.0     0.0       3.58555

"Ā. = QR:"

4×3 Matrix{Float64}:
 -2.0  2.0           3.0
  5.0  4.09145e-17  -1.13425e-16
  5.0  5.0          -2.0
 -4.0  3.0           5.0

"ϵ = Ā - A"

4×3 Matrix{Float64}:
 0.0  0.0           0.0
 0.0  4.09145e-17  -1.13425e-16
 0.0  0.0           0.0
 0.0  0.0           0.0

In [ ]:
#b)Compute the clgs of the 2-by-2 matrix A as in equation (9.1) of your book (page 67).Test orthogonality by computing the matrix norm of the matrix Q^T Q − I_n,
# where I_n is n-by-n identity matrix. Compute both the 2-norm and the Frobenius norm of this matrix, and explain the result.
# Solution: using the fucntion clgs computed in a) we have:

A = Float64.([0.70000    0.70711;
     0.70001    0.70711])
display(A)
Q, R = clgs(A)
Ā = Q * R

display("Q")
display(Q)
display("R")
display(R)
display("Reconstruction → Ā = QR")
display(Ā)

#Testing Orthogonality we have:
Iₙ = I(2)
ϵ = Q'Q - Iₙ
display("ϵ =Q'Q - Iₙ ")
display(ϵ)

F = norm(ϵ)
L = opnorm(ϵ)

display("F-Norm:")
display(F)

display("L-2 Norm:")
display(L)

q₁ = Q[:,1]
q₂ = Q[:,2]

display(q₁)
display(q₂)

cosθ = (q₁ ⋅ q₂) / (norm(q₁) * norm(q₂))
θ = acosd(clamp(cosθ, -1, 1))
display(θ)
display(90 - θ)


2×2 Matrix{Float64}:
 0.7      0.70711
 0.70001  0.70711

"Q"

2×2 Matrix{Float64}:
 0.707102   0.707112
 0.707112  -0.707102

"R"

2×2 Matrix{Float64}:
 0.989957  1.0
 0.0       7.14284e-6

...

2.3014368188967183e-11

2-element Vector{Float64}:
 0.7071017304418633
 0.7071118318951554

2-element Vector{Float64}:
  0.7071118319114289
 -0.7071017304255895

89.99999999868137

1.318625209023594e-9

#### (b) Explanation:

Here see that $||\epsilon||_2 = 2.3014368188967183e-11$ and $||\epsilon||_F = 3.2547231622285317e-11$. Both norms are non-zero up to degree $10^{-11}$, indicating a non-zero error matrix $\epsilon = Q'Q -I$. However, as per the properties of orthonormal matrices, we know $Q^*Q = I \quad \Rightarrow Q^*Q - I = 0 \quad \Rightarrow ||Q^*Q - I|| = ||0|| = 0$.\\

See that the diagonal entries of $\epsilon$ are zero with non-zero entries on the off-diagonal entries, indicating $q_1, q_2 \text{ of } Q$ computed through the QR decomp are not perfectly orthogonal, the angle between them can be computed as:
$$\cos^{-1}(\frac{q_1^* q_2}{||q_1||^* ||q_2||}) = 89.99999999868137$$
showcasing an error of $1.318625209023594e-9$. The columns of $Q$ have unit length to machine precision, but they are **not exactly orthogonal**.

During each step of the clgs computation, floating point arithmetic rounds every operation to about $10^{-16}$ relative accuracy, which compounds as the algorithm processes the next column. However, our orthogonality error is about $10^{-11}$ showing an amplification of about $10^{5}$. This is likely because entries of A are perturbations of around the same number $0.7$, which cancels out columns, introducing floating point numbers during the computation process.

### Problem 2

In [38]:
# Modified Gram Schmidt (MGS) iteration
# Implement the mgs algorithm for an m-by-n real matrix A, m ≥ n; the pseudo-code
# is given in Algorithm 8.1. The function should output matrices Q and R.
function mgs(A::AbstractMatrix{<:Real})
    m, n = size(A)
    V = Float64.(copy(A))
    Q = zeros(Float64, m,n)
    R = zeros(Float64, n,n)
    for i in 1:n
        R[i,i] = norm(V[:, i])
        Q[:, i] = V[:,i]/ R[i,i]
        for j in (i+1):n
            R[i,j] = dot(Q[:, i], V[:, j])
            V[:, j] -= R[i,j]* Q[:,i]
        end
    end
    return Q, R
end


mgs (generic function with 2 methods)

In [39]:
# HouseHolder Triangularization
# b) Implement the house algorithm for an m-by-n real matrix A, m ≥ n; the pseudo-code
# is given in Algorithm 10.1 (follow the book’s advice about using Algorithm 10.3 to
# construct Q). The function should output matrices Q and R.

function house(A::AbstractMatrix{<:Real})
    m,n = size(A)
    R = Float64.(copy(A))
    W = zeros(Float64, m, n)
    for k in 1:n
        x = R[k:m, k]
        v = copy(x)
        v[1] += (x[1] >= 0 ? 1 : -1) * norm(x)
        v /= norm(v, 2)
        R[k:m, k:n] -= 2 * v * (v' * R[k:m, k:n])
        W[k:m,k] = v
    end
    Q = formQ(W)
    return Q, triu(R)
end

function formQ(W::AbstractMatrix{<:Real})
    m, n = size(W)
    Q = Matrix{Float64}(I, m, m)
    for j in 1:m
        x = Q[:, j]
        for k in n:-1:1
            v = W[k:m, k]
            x[k:m] -= 2 * v * (v' * x[k:m])
        end
        Q[:, j] = x
    end
    return Q
end

formQ (generic function with 1 method)

In [40]:
# c) Implement the givens algorithm you derived in exercise 10.4. The function should
# output matrices Q and R
# Solution: for each column j, zero out the entries below the diagonal from the bottom up, using a rotation
# G = [c s; -s c] on rows (i-1, i) with c = a/r, s = b/r, r = √(a² + b²), which maps (a, b) → (r, 0).

function givens(A::AbstractMatrix{<:Real})
    m, n = size(A)
    R = Float64.(copy(A))
    Q = Matrix{Float64}(I, m, m)
    for j in 1:n
        for i in m:-1:(j+1)
            a, b = R[i-1, j], R[i, j]
            b == 0 && continue
            r = hypot(a, b)
            c, s = a / r, b / r
            G = [c s; -s c]
            R[[i-1, i], j:n] = G * R[[i-1, i], j:n]
            Q[:, [i-1, i]] = Q[:, [i-1, i]] * G'
        end
    end
    return Q, triu(R)
end

givens (generic function with 1 method)

In [41]:
# d) Creating A = Q_0 R_0, to know true matrices that develop A then testing
# on its implementation.

function make_Q0(m::Int, n::Int)
    F = qr(randn(m, m))
    Qfull = Matrix(F.Q) * Diagonal(sign.(diag(F.R)))
    return Qfull[:, 1:n]
end

function make_R0(n::Int; σ = 0.1)
    S0 = Diagonal([2.0^(-j) for j in 1:n])
    R = triu(σ * randn(n, n), 1) + I
    return S0 * R
end

function make_A(m::Int, n::Int)
    Q0 = make_Q0(m, n)
    R0 = make_R0(n)
    A = Q0 * R0
    return A, Q0, R0
end

make_A (generic function with 1 method)

In [46]:
A, Q0, R0= make_A(40,40)

CLQ, CLR = clgs(A)
MGQ, MGR = mgs(A)
HOQ, HOR = house(A)
GivQ, GivR = givens(A)
F = qr(A)
Q_lapack = Matrix(F.Q)
R_lapack = F.R

display("Original Matrix Decomp")
display(A)
display(Q0)
display(R0)

display("clgs, Q, R in order")
display(CLQ)
display(CLR)

display("mgs, Q, R in order")
display(MGQ)
display(MGR)

display("house, Q, R in order")
display(HOQ)
display(HOQ)

display("givens, Q, R in order")
display(GivQ)
display(GivR)

display("default julia Q, R")
display(Q_lapack)
display(R_lapack)



"Original Matrix Decomp"

40×40 Matrix{Float64}:
 -0.0741163     0.0454087    0.00889346   …   3.75871e-5    0.0071409
  0.0102389     0.00374528   0.00648589      -0.00280083   -0.00193592
  0.0425081     0.0524618   -0.0144594        0.00647406   -0.00728833
 -0.0727837     0.0182243   -0.0264678        0.0030573     0.0149866
 -0.0359671    -0.0283164   -0.005355         0.000361815   0.00667309
 -0.00829652    0.107632    -0.00741055   …   0.00485126   -0.00265731
 -0.0469708     0.0322331    0.00202423       0.00268582    0.00373989
 -0.0288803    -0.00780374   0.0108658        0.000952911   0.0010873
  0.197847     -0.0547622   -0.00940695       0.00356165   -0.0271073
 -0.000121915   0.0871111   -0.00866619       0.003212     -0.00171835
  ⋮                                       ⋱                
 -0.174574      0.0357514   -0.0157777       -0.000431843   0.0287099
  0.00298609    0.002775    -0.0121852       -0.000384941   0.00246168
 -0.0676199     0.015693    -0.00797604      -0.00152091    0.0114506


40×40 Matrix{Float64}:
 -0.148233     0.166117    0.0889048   …   0.178966    0.290311   -0.0278655
  0.0204777    0.0171248   0.0525456       0.236189   -0.112643   -0.136187
  0.0850161    0.218747   -0.101476        0.156731   -0.0640907   0.00799865
 -0.145567     0.0576586  -0.202494        0.186498   -0.0882248   0.100843
 -0.0719342   -0.120796   -0.0498617      -0.0863456   0.050756   -0.139918
 -0.016593     0.428792   -0.0254461   …   0.0757392  -0.0132654   0.0234835
 -0.0939416    0.119098    0.0285206      -0.112726   -0.175889    0.00107785
 -0.0577605   -0.0372616   0.085926        0.205477   -0.153719   -0.245146
  0.395695    -0.177625   -0.102017       -0.0970857   0.0109096   0.348829
 -0.00024383   0.348419   -0.0422679       0.034085   -0.178624    0.0358176
  ⋮                                    ⋱                          
 -0.349149     0.106455   -0.106512        0.0473834   0.0451278   0.141748
  0.00597218   0.0117252  -0.0967672      -0.102745    0.0170142  -

40×40 Matrix{Float64}:
 0.5  -0.0261713   0.0040971    0.0297823  …   0.00217047   -0.0740635
 0.0   0.25       -0.00970587  -0.0361853      0.00781983   -0.00921945
 0.0   0.0         0.125        0.0145813     -0.0159708    -0.0225897
 0.0   0.0         0.0          0.0625        -0.0129273     0.00583639
 0.0   0.0         0.0          0.0            0.00342402   -0.00369976
 0.0   0.0         0.0          0.0        …   2.03169e-5    0.000276774
 0.0   0.0         0.0          0.0           -0.0015298     0.000596784
 0.0   0.0         0.0          0.0            0.000109703  -0.000390689
 0.0   0.0         0.0          0.0            7.03359e-5   -1.12466e-5
 0.0   0.0         0.0          0.0            8.43798e-5    7.37789e-5
 ⋮                                         ⋱                
 0.0   0.0         0.0          0.0            4.54685e-11   3.42486e-11
 0.0   0.0         0.0          0.0            6.97484e-12   1.1922e-11
 0.0   0.0         0.0          0.0           -2.1

"clgs, Q, R in order"

...

40×40 Matrix{Float64}:
 -0.148233     0.166117    0.0889048   …   0.178966    0.290311    0.0278656
  0.0204777    0.0171248   0.0525456       0.236189   -0.112643    0.136187
  0.0850161    0.218747   -0.101476        0.156731   -0.0640907  -0.00799844
 -0.145567     0.0576586  -0.202494        0.186498   -0.0882248  -0.100843
 -0.0719342   -0.120796   -0.0498617      -0.0863456   0.0507561   0.139918
 -0.016593     0.428792   -0.0254461   …   0.0757392  -0.0132655  -0.0234835
 -0.0939416    0.119098    0.0285206      -0.112726   -0.175889   -0.00107785
 -0.0577605   -0.0372616   0.085926        0.205477   -0.15372     0.245146
  0.395695    -0.177625   -0.102017       -0.0970856   0.0109098  -0.348829
 -0.00024383   0.348419   -0.0422679       0.0340849  -0.178624   -0.0358175
  ⋮                                    ⋱                          
 -0.349149     0.106455   -0.106512        0.0473835   0.0451278  -0.141748
  0.00597218   0.0117252  -0.0967672      -0.102745    0.0170141   

40×40 Matrix{Float64}:
 0.5  -0.0261713   0.0040971    0.0297823  …   0.00217047   -0.0740635
 0.0   0.25       -0.00970587  -0.0361853      0.00781983   -0.00921945
 0.0   0.0         0.125        0.0145813     -0.0159708    -0.0225897
 0.0   0.0         0.0          0.0625        -0.0129273     0.00583639
 0.0   0.0         0.0          0.0            0.00342402   -0.00369976
 0.0   0.0         0.0          0.0        …   2.03169e-5    0.000276774
 0.0   0.0         0.0          0.0           -0.0015298     0.000596784
 0.0   0.0         0.0          0.0            0.000109703  -0.000390689
 0.0   0.0         0.0          0.0            7.03359e-5   -1.12466e-5
 0.0   0.0         0.0          0.0            8.43798e-5    7.37789e-5
 ⋮                                         ⋱                
 0.0   0.0         0.0          0.0            4.54685e-11   3.42486e-11
 0.0   0.0         0.0          0.0            6.97484e-12   1.19219e-11
 0.0   0.0         0.0          0.0           -2.

"default julia Q, R"

40×40 Matrix{Float64}:
 -0.148233    -0.166117    0.0889048   …   0.178966   -0.290311    0.0278648
  0.0204777   -0.0171248   0.0525456       0.236189    0.112642    0.136187
  0.0850161   -0.218747   -0.101476        0.156731    0.0640908  -0.00799826
 -0.145567    -0.0576586  -0.202494        0.186498    0.0882252  -0.100843
 -0.0719342    0.120796   -0.0498617      -0.0863456  -0.0507564   0.139918
 -0.016593    -0.428792   -0.0254461   …   0.0757391   0.0132655  -0.0234835
 -0.0939416   -0.119098    0.0285206      -0.112726    0.175889   -0.00107709
 -0.0577605    0.0372616   0.085926        0.205477    0.153719    0.245146
  0.395695     0.177625   -0.102017       -0.0970857  -0.0109088  -0.348829
 -0.00024383  -0.348419   -0.0422679       0.034085    0.178624   -0.035817
  ⋮                                    ⋱                          
 -0.349149    -0.106455   -0.106512        0.0473835  -0.0451273  -0.141749
  0.00597218  -0.0117252  -0.0967672      -0.102745   -0.0170143   0

40×40 Matrix{Float64}:
 0.5  -0.0261713  0.0040971    0.0297823  …   0.00217047   -0.0740635
 0.0  -0.25       0.00970587   0.0361853     -0.00781983    0.00921945
 0.0   0.0        0.125        0.0145813     -0.0159708    -0.0225897
 0.0   0.0        0.0         -0.0625         0.0129273    -0.00583639
 0.0   0.0        0.0          0.0           -0.00342402    0.00369976
 0.0   0.0        0.0          0.0        …   2.03169e-5    0.000276774
 0.0   0.0        0.0          0.0           -0.0015298     0.000596784
 0.0   0.0        0.0          0.0            0.000109703  -0.000390689
 0.0   0.0        0.0          0.0           -7.03359e-5    1.12466e-5
 0.0   0.0        0.0          0.0            8.43798e-5    7.37789e-5
 ⋮                                        ⋱                
 0.0   0.0        0.0          0.0           -4.54685e-11  -3.42485e-11
 0.0   0.0        0.0          0.0           -6.97483e-12  -1.19219e-11
 0.0   0.0        0.0          0.0           -2.11519e-12   9.

In [ ]:
# e) evaluating the norm values.

function evaluate(Q:: AbstractMatrix)
    m,n = size(Q)
    err = Q'Q - I(m)
    l2 = opnorm(err,2)
    f = norm(err)
    return l2, f
end

clgl2, clgf = evaluate(CLQ)
mgsl2, mgsf = evaluate(MGQ)
hol2, hof = evaluate(HOQ)
givl2, givf = evaluate(GivQ)
defl2, deff = evaluate(Q_lapack)

display("CLGS")
display("l2 norm")
display(clgl2)
display("Frobenius")
display(clgf)


display("MGS")
display("l2 norm")
display(mgsl2)
display("Frobenius")
display(mgsf)


display("HouseHolder")
display("l2 norm")
display(hol2)
display("Frobenius")
display(hof)


display("Givens")
display("l2 norm")
display(hol2)
display("Frobenius")
display(hof)

display("Julia Default")
display("l2 norm")
display(defl2)
display("Frobenius")
display(deff)

"CLGS"

"l2 norm"

6.742434191571925

"Frobenius"

7.576988502067704

...

"Julia Default"

"l2 norm"

2.1088451720026343e-15

"Frobenius"

5.34483139134779e-15

#### (e) Explanation:

Here we measure $\epsilon = Q^TQ - I$ with the 2-norm $||\epsilon||_2$ (the largest amount by which $Q$ can stretch or shrink a vector) and the Frobenius norm $||\epsilon||_F$ (the total size of all entries of $\epsilon$). If $Q$ were perfectly orthogonal, we know $Q^TQ = I \quad \Rightarrow ||Q^TQ - I|| = 0$ in both norms. Our matrix $A$ is built in a way that its columns are very close to being linearly dependent, which makes this a hard test for the algorithms.

**CLGS: Explanation**

See that $||\epsilon||_2 = 6.74$ and $||\epsilon||_F = 7.58$, both bigger than 1, indicating that the $Q$ computed by clgs has completely lost orthogonality, i.e. many of its columns point in almost the same direction. As in Problem 1(b), each step subtracts nearly parallel vectors and divides by a tiny $r_{jj}$, which blows up the rounding error, and since clgs always projects against the original column $a_j$, these errors are never removed and keep compounding as the algorithm moves to the next column.

**MGS: Explanation**

Here $||\epsilon||_2 = 1.35 \times 10^{-5}$ and $||\epsilon||_F = 1.98 \times 10^{-5}$, which is much better than clgs, since mgs projects against the already updated vector and removes the earlier rounding errors along the way. However, this is still around ten orders of magnitude above machine precision ($\approx 10^{-16}$).

**House, Givens and Julia Default: Explanation**

House gives $||\epsilon||_2 = 3.26 \times 10^{-15}$ and $||\epsilon||_F = 6.97 \times 10^{-15}$, Julia's default qr gives $||\epsilon||_2 = 2.11 \times 10^{-15}$ and $||\epsilon||_F = 5.34 \times 10^{-15}$, and the Givens rotation gives $||\epsilon||_2 = 3.26 \times 10^{-15}$ and $||\epsilon||_F =6.97 \times 10^{-15}$ all at the level of machine precision. This is because these methods never build $Q$ by subtracting projections, instead $Q$ is a product of reflections (house, Julia default) or rotations (givens), each of which is exactly orthogonal, so $Q$ stays orthogonal no matter how ill-conditioned $A$ is.

**2-norm vs. Frobenius norm**

In every case $||\epsilon||_2 \leq ||\epsilon||_F$, as expected. For clgs the two norms are close ($6.74$ vs. $7.58$), showing that its error is concentrated in only a few directions, while for house and Julia's default the Frobenius norm is about $2$–$2.5$ times larger, showing that their tiny rounding errors are spread out over many directions. Either way, both norms give the same ranking: clgs is by far the worst, mgs is in between, and house, givens and Julia's default qr are orthogonal up to machine precision.


In [55]:
# f) Evaluate the errors of Q and R compared to the ground truth {Q₀, R₀} for the different algorithms.
# Compute 20 random realizations of A and report the mean and standard deviation of the error.

N = 20
Qerr_clgs = zeros(N);  Rerr_clgs = zeros(N)
Qerr_mgs = zeros(N);  Rerr_mgs = zeros(N)
Qerr_house = zeros(N);  Rerr_house = zeros(N)
Qerr_givens = zeros(N);  Rerr_givens = zeros(N)
Qerr_lapack = zeros(N);  Rerr_lapack = zeros(N)

Random.seed!(561)
for t in 1:N
    At, Q0t, R0t = make_A(40, 40)

    # CLGS
    Qt, Rt = clgs(At)
    D = Diagonal(sign.(diag(Rt)))
    Qt = Qt * D;  Rt = D * Rt
    Qerr_clgs[t] = opnorm(Qt - Q0t);  Rerr_clgs[t] = opnorm(Rt - R0t)

    # MGS
    Qt, Rt = mgs(At)
    D = Diagonal(sign.(diag(Rt)))
    Qt = Qt * D;  Rt = D * Rt
    Qerr_mgs[t] = opnorm(Qt - Q0t);  Rerr_mgs[t] = opnorm(Rt - R0t)

    # HOUSE
    Qt, Rt = house(At)
    D = Diagonal(sign.(diag(Rt)))
    Qt = Qt * D;  Rt = D * Rt
    Qerr_house[t] = opnorm(Qt - Q0t);  Rerr_house[t] = opnorm(Rt - R0t)

    # GIVENS
    Qt, Rt = givens(At)
    D = Diagonal(sign.(diag(Rt)))
    Qt = Qt * D;  Rt = D * Rt
    Qerr_givens[t] = opnorm(Qt - Q0t);  Rerr_givens[t] = opnorm(Rt - R0t)

    # LAPACK qr
    Ft = qr(At)
    Qt = Matrix(Ft.Q);  Rt = Matrix(Ft.R)
    D = Diagonal(sign.(diag(Rt)))
    Qt = Qt * D;  Rt = D * Rt
    Qerr_lapack[t] = opnorm(Qt - Q0t);  Rerr_lapack[t] = opnorm(Rt - R0t)
end

display("CLGS")
display("mean ‖Q − Q₀‖₂"); display(mean(Qerr_clgs))
display("std ‖Q − Q₀‖₂");  display(std(Qerr_clgs))
display("mean ‖R − R₀‖₂"); display(mean(Rerr_clgs))
display("std ‖R − R₀‖₂");  display(std(Rerr_clgs))

display("MGS")
display("mean ‖Q − Q₀‖₂"); display(mean(Qerr_mgs))
display("std ‖Q − Q₀‖₂");  display(std(Qerr_mgs))
display("mean ‖R − R₀‖₂"); display(mean(Rerr_mgs))
display("std ‖R − R₀‖₂");  display(std(Rerr_mgs))

display("HOUSE")
display("mean ‖Q − Q₀‖₂"); display(mean(Qerr_house))
display("std ‖Q − Q₀‖₂");  display(std(Qerr_house))
display("mean ‖R − R₀‖₂"); display(mean(Rerr_house))
display("std ‖R − R₀‖₂");  display(std(Rerr_house))

display("GIVENS")
display("mean ‖Q − Q₀‖₂"); display(mean(Qerr_givens))
display("std ‖Q − Q₀‖₂");  display(std(Qerr_givens))
display("mean ‖R − R₀‖₂"); display(mean(Rerr_givens))
display("std ‖R − R₀‖₂");  display(std(Rerr_givens))

display("LAPACK qr")
display("mean ‖Q − Q₀‖₂"); display(mean(Qerr_lapack))
display("std ‖Q − Q₀‖₂");  display(std(Qerr_lapack))
display("mean ‖R − R₀‖₂"); display(mean(Rerr_lapack))
display("std ‖R − R₀‖₂");  display(std(Rerr_lapack))


"CLGS"

"mean ‖Q − Q₀‖₂"

3.005642837812532

"std ‖Q − Q₀‖₂"

0.21671527260846546

...

2.4481040741711295e-6

"mean ‖R − R₀‖₂"

1.465787487536878e-16

"std ‖R − R₀‖₂"

4.6885065513572507e-17

#### (f) Explanation:

Here we compare the $Q$ and $R$ computed by each algorithm with the true factors $Q_0, R_0$, by computing $||Q - Q_0||_2$ and $||R - R_0||_2$ over 20 random matrices $A$ of size $40 \times 40$, and report the mean and standard deviation of each error. Before subtracting, the signs of $Q$ and $R$ are flipped so that the diagonal of $R$ is positive, since the QR factorization is only unique with a positive diagonal, otherwise $Q$ and $Q_0$ could differ just by the signs of their columns.

**CLGS: Explanation**

See that $||Q - Q_0||_2 = 3.01 \pm 0.22$, indicating that the $Q$ computed by clgs has nothing to do with the true $Q_0$. Since both $Q$ and $Q_0$ have columns of length 1, an error of about 3 means that the columns are pointing in completely different directions, which matches the complete loss of orthogonality we saw in (e). $R$ is also off, with $||R - R_0||_2 = 1.76 \times 10^{-8} \pm 1.22 \times 10^{-8}$, which is about eight orders of magnitude worse than the other methods.

**MGS: Explanation**

Here $||Q - Q_0||_2 = 2.01 \times 10^{-5} \pm 1.05 \times 10^{-5}$, which is about the same size as the orthogonality error of mgs in (e), showing that most of the error in $Q$ comes from $Q$ not being orthogonal. $R$ on the other hand is accurate up to machine precision, with $||R - R_0||_2 = 1.40 \times 10^{-16} \pm 4.89 \times 10^{-17}$.

**House, Givens and Julia Default QR: Explanation**

All three give $R$ accurate up to machine precision ($||R - R_0||_2 \approx 1.5$–$1.9 \times 10^{-16}$). However, see that even though their $Q$ is orthogonal up to machine precision (from (e)), $||Q - Q_0||_2$ is still around $10^{-6}$ (house $3.05 \times 10^{-6} \pm 3.80 \times 10^{-6}$, givens $1.81 \times 10^{-6} \pm 8.82 \times 10^{-7}$, Julia Default QR $4.80 \times 10^{-6} \pm 2.45 \times 10^{-6}$), which is about ten orders of magnitude above machine precision. This is not a fault of the algorithms. Since the diagonal of $R_0$ shrinks all the way down to $2^{-40} \approx 10^{-12}$, the last columns of $A$ are very close to being linear combinations of the earlier ones, so the directions of the last columns of $Q$ are decided by extremely small differences in $A$. As a result, even the tiny rounding errors made by these algorithms are enough to visibly change $Q$. However, the errors in $Q$ and $R$ are tied together such that they cancel out in the product, so $QR$ still gives back $A$ up to machine precision.


In [58]:
# g) Let A be of size m-by-n, n = 40, m = 80, 160, 320, 640. Q₀ consists
# of the first n columns of a random m-by-m orthogonal matrix, while R₀
# is constructed as before. How does this affect the results in (d)?
n = 40
ms = [80, 160, 320, 640]
N = 20
orth_clgs = zeros(4, N);     Qerr_clgs = zeros(4, N);     Rerr_clgs = zeros(4, N)
orth_mgs = zeros(4, N);      Qerr_mgs = zeros(4, N);      Rerr_mgs = zeros(4, N)
orth_house = zeros(4, N);    Qerr_house = zeros(4, N);    Rerr_house = zeros(4, N)
orth_givens = zeros(4, N);   Qerr_givens = zeros(4, N);   Rerr_givens = zeros(4, N)
orth_default = zeros(4, N);  Qerr_default = zeros(4, N);  Rerr_default = zeros(4, N)

Random.seed!(561)
for i in 1:4
    for t in 1:N
        At, Q0t, R0t = make_A(ms[i], n)

        # CLGS
        Qt, Rt = clgs(At)
        D = Diagonal(sign.(diag(Rt)))
        Qt = Qt * D;  Rt = D * Rt
        orth_clgs[i, t] = opnorm(Qt'Qt - I);  Qerr_clgs[i, t] = opnorm(Qt - Q0t);  Rerr_clgs[i, t] = opnorm(Rt - R0t)

        # MGS
        Qt, Rt = mgs(At)
        D = Diagonal(sign.(diag(Rt)))
        Qt = Qt * D;  Rt = D * Rt
        orth_mgs[i, t] = opnorm(Qt'Qt - I);  Qerr_mgs[i, t] = opnorm(Qt - Q0t);  Rerr_mgs[i, t] = opnorm(Rt - R0t)

        # HOUSE (reduced: first n columns of Q, first n rows of R)
        Qt, Rt = house(At)
        Qt = Qt[:, 1:n];  Rt = Rt[1:n, :]
        D = Diagonal(sign.(diag(Rt)))
        Qt = Qt * D;  Rt = D * Rt
        orth_house[i, t] = opnorm(Qt'Qt - I);  Qerr_house[i, t] = opnorm(Qt - Q0t);  Rerr_house[i, t] = opnorm(Rt - R0t)

        # GIVENS (reduced: first n columns of Q, first n rows of R)
        Qt, Rt = givens(At)
        Qt = Qt[:, 1:n];  Rt = Rt[1:n, :]
        D = Diagonal(sign.(diag(Rt)))
        Qt = Qt * D;  Rt = D * Rt
        orth_givens[i, t] = opnorm(Qt'Qt - I);  Qerr_givens[i, t] = opnorm(Qt - Q0t);  Rerr_givens[i, t] = opnorm(Rt - R0t)

        # Default qr (Matrix(Ft.Q) already gives the reduced m×n Q, like qr(A, "econ"))
        Ft = qr(At)
        Qt = Matrix(Ft.Q);  Rt = Matrix(Ft.R)
        D = Diagonal(sign.(diag(Rt)))
        Qt = Qt * D;  Rt = D * Rt
        orth_default[i, t] = opnorm(Qt'Qt - I);  Qerr_default[i, t] = opnorm(Qt - Q0t);  Rerr_default[i, t] = opnorm(Rt - R0t)
    end
end

# Each displayed vector lists the mean over the 20 matrices for m = 80, 160, 320, 640 (top to bottom).
display("CLGS")
display("mean ‖QᵀQ − I‖₂");  display(vec(mean(orth_clgs, dims=2)))
display("mean ‖Q − Q₀‖₂");   display(vec(mean(Qerr_clgs, dims=2)))
display("mean ‖R − R₀‖₂");   display(vec(mean(Rerr_clgs, dims=2)))

display("MGS")
display("mean ‖QᵀQ − I‖₂");  display(vec(mean(orth_mgs, dims=2)))
display("mean ‖Q − Q₀‖₂");   display(vec(mean(Qerr_mgs, dims=2)))
display("mean ‖R − R₀‖₂");   display(vec(mean(Rerr_mgs, dims=2)))

display("HOUSE")
display("mean ‖QᵀQ − I‖₂");  display(vec(mean(orth_house, dims=2)))
display("mean ‖Q − Q₀‖₂");   display(vec(mean(Qerr_house, dims=2)))
display("mean ‖R − R₀‖₂");   display(vec(mean(Rerr_house, dims=2)))

display("GIVENS")
display("mean ‖QᵀQ − I‖₂");  display(vec(mean(orth_givens, dims=2)))
display("mean ‖Q − Q₀‖₂");   display(vec(mean(Qerr_givens, dims=2)))
display("mean ‖R − R₀‖₂");   display(vec(mean(Rerr_givens, dims=2)))

display("Julia_Def qr")
display("mean ‖QᵀQ − I‖₂");  display(vec(mean(orth_default, dims=2)))
display("mean ‖Q − Q₀‖₂");   display(vec(mean(Qerr_default, dims=2)))
display("mean ‖R − R₀‖₂");   display(vec(mean(Rerr_default, dims=2)))


"CLGS"

"mean ‖QᵀQ − I‖₂"

4-element Vector{Float64}:
 8.699809484153272
 8.731363226108169
 8.778407198258938
 9.22341570025842

"mean ‖Q − Q₀‖₂"

4-element Vector{Float64}:
 3.0035181973237544
 3.0286319234244856
 3.0268009761561485
 3.0906407954331323

...

4-element Vector{Float64}:
 1.496175263821287e-15
 1.4908908990858493e-15
 1.810420309492369e-15
 2.0543940863609305e-15

"mean ‖Q − Q₀‖₂"

4-element Vector{Float64}:
 3.575509192887893e-5
 4.963237280227533e-5
 6.237048463569745e-5
 5.907646973731252e-5

"mean ‖R − R₀‖₂"

4-element Vector{Float64}:
 1.4490413127059915e-16
 1.7159892720879369e-16
 2.4429556782381375e-16
 2.6004870277509334e-16

#### (g) Explanation:

Here we repeat the experiment from (d)–(f) for tall matrices with $n = 40$ and $m = 80, 160, 320, 640$, using the reduced QR, and report the mean over 20 random matrices for each $m$. Note that $R_0$ is built exactly as before, so the columns of $A$ are just as close to being linearly dependent as in (d), the only difference is that each column is now longer.

**CLGS: Explanation**

Nothing changes compared to (d). For every $m$ we still get $||Q^TQ - I||_2 \approx 8.7$–$9.2$ and $||Q - Q_0||_2 \approx 3.0$, so $Q$ has still completely lost orthogonality and has nothing to do with $Q_0$, and $||R - R_0||_2$ stays around $1.6$–$2.5 \times 10^{-8}$, same as in the square case.

**MGS: Explanation**

The loss of orthogonality grows slightly with $m$, from $2.5 \times 10^{-5}$ at $m = 80$ to $4.5 \times 10^{-5}$ at $m = 640$ (compared to $2.0 \times 10^{-5}$ in the square case), and $||Q - Q_0||_2$ follows it closely ($3.1 \times 10^{-5}$ to $4.9 \times 10^{-5}$), showing again that the error in $Q$ mostly comes from $Q$ not being orthogonal. This slow growth is because longer columns mean more arithmetic in every projection, and hence a bit more rounding error. However, $R$ stays accurate up to machine precision ($1.3$–$2.3 \times 10^{-16}$).

**House, Givens and Julia Default QR: Explanation**

$Q$ stays orthogonal up to machine precision for every $m$, and $R$ stays accurate up to machine precision ($||R - R_0||_2 \approx 1.4$–$5.6 \times 10^{-16}$). For house and givens, both errors creep up slowly with $m$ for the same reason as mgs, e.g. the orthogonality error of house goes from $2.3 \times 10^{-15}$ at $m = 80$ to $4.3 \times 10^{-15}$ at $m = 640$, while Julia Default QR stays the flattest ($1.5$–$2.1 \times 10^{-15}$) and has the smallest orthogonality error at every $m$.

The one noticeable change compared to (d) is in $||Q - Q_0||_2$, which jumps about ten times from the square case (house $3.05 \times 10^{-6}$, givens $1.81 \times 10^{-6}$, Julia Default QR $4.80 \times 10^{-6}$) to around $2$–$4 \times 10^{-5}$ already at $m = 80$, and then grows slowly up to around $5$–$6 \times 10^{-5}$ at $m = 640$. When $A$ is square, $Q$ is an $m \times m$ orthogonal matrix, so its columns already span the whole space and rounding errors can only rotate the columns among themselves. When $m > n$, the columns of $Q$ only span an $n$-dimensional subspace of $\mathbb{R}^m$, so rounding errors can now also tilt this subspace into the extra $m - n$ directions, which makes $Q$ more sensitive to rounding errors for tall matrices.

